# Agente basado en LLM + enriquecimiento semántico de avisos inmobiliarios

**Trabajo Final — Deep Learning**  
Maestría en Management & Analytics (MMA), ITBA  
Alumno: **Joaquín Héctor Vassarotto** — Legajo 106442

---

## El problema

Construir modelos de valuación inmobiliaria (AVM) en CABA choca con dos limitaciones de datos:

1. **Adquirir avisos es frágil y manual.** Los scrapers por reglas fijas se rompen ante cada cambio de layout del portal.
2. **La información de valor está en el texto libre.** Estado, amenities, orientación, antigüedad y señales del vendedor viven en la descripción, no en los campos tabulares.

## La solución: dos capas

| Capa | Qué hace | Se entrena? |
|---|---|---|
| **1. Agente (orquestación)** | Un LLM local con patrón ReAct navega ZonaProp, decide cómo paginar y qué extraer, y se recupera ante errores | No — el LLM se usa pre-entrenado |
| **2. Enriquecimiento (NLP)** | Dos transformers BETO fine-tuneados convierten la descripción en variables estructuradas | **Sí — es el núcleo de la materia** |

> El componente evaluado como Deep Learning es la **capa 2**. La capa 1 construye el dataset.

In [1]:
import json, sys, os
from pathlib import Path

# Permite correr el notebook desde notebooks/ o desde la raiz del repo
ROOT = Path.cwd()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
os.chdir(ROOT)

from src.utils.config import load_config
from src.utils.io import read_jsonl

cfg = load_config()
print("Repo:", ROOT)
print("LLM del agente :", cfg["agent"]["model"])
print("Modelo base NLP:", cfg["ner"]["base_model"])

Repo: C:\Users\Usuario\Desktop\zonaprop-agent-nlp
LLM del agente : llama3.2:3b-instruct-q4_K_M
Modelo base NLP: dccuchile/bert-base-spanish-wwm-cased


---
# Capa 1 — El agente

El LLM actúa como *policy*: razona sobre el estado de la página y decide la próxima acción.
Las herramientas que tiene disponibles son las que le permiten interactuar con el navegador.

In [2]:
from src.agent.react_agent import SYSTEM_PROMPT

print(SYSTEM_PROMPT)

Sos un agente que recolecta avisos de departamentos en venta en CABA desde ZonaProp.
Tu objetivo es recorrer el listado pagina por pagina y extraer los avisos.

Herramientas disponibles:
- goto_search_page(page_number): navega a una pagina del listado.
- extract_current_page(): extrae y guarda los avisos de la pagina actual.
- has_next_page(): indica si hay pagina siguiente.

Estrategia:
1. Empeza en la pagina 1 con goto_search_page.
2. Llama extract_current_page para guardar los avisos.
3. Usa has_next_page; si hay siguiente, avanza; si no, termina.
4. Si una accion devuelve ERROR, reintenta esa pagina una vez antes de continuar.
5. No repitas paginas ya procesadas. Se conciso en tu razonamiento.

Cuando termines o alcances el limite de paginas, responde 'LISTO'.


## Métrica 1 del agente: tasa de éxito de extracción

De los avisos que el agente **detecta** en una página de listado, ¿cuántos termina extrayendo
con una descripción utilizable? Se registra durante el scrape real.

In [3]:
metricas = sorted(Path("reports").glob("agent_metrics*.json"))
if metricas:
    for m in metricas:
        d = json.loads(m.read_text(encoding="utf-8"))
        print(f"--- {m.name}  (modo: {d.get('mode')}) ---")
        for k, v in d["summary"].items():
            print(f"  {k:28s} {v}")
else:
    print("Todavia no se corrio el scrape real (reports/agent_metrics_*.json no existe).")
    print("Para generarlo:  python -m src.agent.run_scrape --mode deterministic --max-listings 300")

--- agent_metrics_argenprop.json  (modo: argenprop) ---
  pages_visited                8
  listings_detected            80
  listings_extracted           40
  extraction_success_rate      0.5
  worst_page_success_rate      0.0
  errors                       0
--- agent_metrics_deterministic.json  (modo: deterministic) ---
  pages_visited                2
  listings_detected            30
  listings_extracted           25
  extraction_success_rate      0.8333
  worst_page_success_rate      0.8333
  errors                       0
--- agent_metrics_grid.json  (modo: grid) ---
  pages_visited                1
  listings_detected            30
  listings_extracted           25
  extraction_success_rate      0.8333
  worst_page_success_rate      0.8333
  errors                       8


## Métrica 2 del agente: robustez ante cambios de layout

Este es el punto que motiva el proyecto. `parser.py` está escrito en **dos niveles**:
primero intenta los selectores `data-qa` del portal y, si fallan, cae a expresiones regulares
sobre el texto plano.

Para medir cuánto aporta ese segundo nivel, tomamos un HTML que el parser sabe leer y le
aplicamos degradaciones que imitan cambios reales del sitio, midiendo qué campos sobreviven.

In [4]:
from src.agent.robustness import evaluate_fixtures

rob = evaluate_fixtures(ROOT)
print(f"Fuente: {rob['fuente']}\n")
print(f"{'Variante del HTML':26s} {'Retencion de campos':>20s}")
print("-" * 48)
for variante, tasa in rob["retencion_promedio"].items():
    valor = f"{tasa:.1%}" if tasa is not None else "(sin datos)"
    print(f"{variante:26s} {valor:>20s}")

Fuente: muestra sintetica (los fixtures guardados no tienen contenido parseable)

Variante del HTML           Retencion de campos
------------------------------------------------
original                                 100.0%
sin_data_qa                               87.5%
renombrado_a_clases                      100.0%
sin_ningun_atributo                       87.5%


**Lectura del resultado.** Cuando desaparecen los selectores `data-qa`, los campos numéricos
(precio, ambientes, dormitorios, baños, antigüedad, expensas) y el barrio se siguen recuperando
por regex sobre el texto plano. Lo único que se pierde es la **descripción**, que necesita al
menos un gancho de markup reconocible para poder aislarse del resto de la página.

Un scraper de un solo nivel habría caído a 0% en ese escenario.

---
# Capa 2 — Los modelos entrenados

## Esquema de etiquetado

Dos tareas distintas sobre la misma descripción:

- **NER (token classification)** con esquema BIO: dónde, dentro del texto, se menciona cada atributo.
- **Clasificación multilabel**: qué señales del vendedor tiene el aviso en conjunto.

Son multilabel y no multiclase porque un aviso puede ser a la vez "dueño directo" y "urgencia".

In [5]:
from src.annotation.label_schema import ENTITY_TYPES, BIO_LABELS, SIGNAL_CLASSES

print("Entidades NER :", ENTITY_TYPES)
print("Etiquetas BIO :", BIO_LABELS)
print("Clases (senal):", SIGNAL_CLASSES)

Entidades NER : ['AMENITY', 'ESTADO', 'ANTIGUEDAD', 'ORIENTACION', 'EXPENSAS']
Etiquetas BIO : ['O', 'B-AMENITY', 'I-AMENITY', 'B-ESTADO', 'I-ESTADO', 'B-ANTIGUEDAD', 'I-ANTIGUEDAD', 'B-ORIENTACION', 'I-ORIENTACION', 'B-EXPENSAS', 'I-EXPENSAS']
Clases (senal): ['DUENO_DIRECTO', 'OPORTUNIDAD', 'URGENCIA', 'REFACCION']


## El dataset

Miramos la distribución de entidades y de clases, porque **condiciona qué métrica tiene sentido**
(volvemos sobre esto más abajo).

In [6]:
from collections import Counter

splits = {}
for s in ["train", "val", "test"]:
    splits[s] = {
        "ner": list(read_jsonl(ROOT / f"data/annotated/ner_{s}.jsonl")),
        "cls": list(read_jsonl(ROOT / f"data/annotated/cls_{s}.jsonl")),
    }
    print(f"{s:6s} NER={len(splits[s]['ner']):5d}   CLS={len(splits[s]['cls']):5d}")

print("\nEntidades en train (conteo de spans):")
ent = Counter(t[2:] for r in splits["train"]["ner"] for t in r["ner_tags"] if t.startswith("B-"))
for k, v in ent.most_common():
    print(f"  {k:14s} {v:6d}")

print("\nSenales en train (un aviso puede tener varias):")
sig = Counter(s for r in splits["train"]["cls"] for s in r.get("signals", []))
n = len(splits["train"]["cls"])
for k, v in sig.most_common():
    print(f"  {k:16s} {v:6d}  ({v/n:.1%} de los avisos)")
sin_senal = sum(1 for r in splits["train"]["cls"] if not r.get("signals"))
print(f"  {'(ninguna)':16s} {sin_senal:6d}  ({sin_senal/n:.1%})")

train  NER=  960   CLS=  960
val    NER=  120   CLS=  120
test   NER=  120   CLS=  120

Entidades en train (conteo de spans):
  AMENITY          2425
  ESTADO            858
  EXPENSAS          794
  ANTIGUEDAD        750
  ORIENTACION       702

Senales en train (un aviso puede tener varias):
  DUENO_DIRECTO       359  (37.4% de los avisos)
  OPORTUNIDAD         291  (30.3% de los avisos)
  REFACCION           257  (26.8% de los avisos)
  URGENCIA            185  (19.3% de los avisos)
  (ninguna)           248  (25.8%)


In [7]:
# Un ejemplo concreto: como se ve una descripcion ya etiquetada
from src.utils.text import group_entities

ej = splits["train"]["ner"][0]
print("TEXTO:\n ", " ".join(ej["tokens"])[:400], "...\n")
print("ENTIDADES ANOTADAS:")
for e in group_entities(list(zip(ej["tokens"], ej["ner_tags"]))):
    print(f"  {e['type']:14s} -> {e['text']}")

TEXTO:
  Excelente departamento de 2 ambientes en el corazón de Caballito . La propiedad da vista abierta al oeste . Unidad a estrenar . Unidad al día con las cuotas y libre deuda . Toilette de recepción y baño completo con bañera . Consultar disponibilidad para visitas y coordinación de horarios . A pocas cuadras del subte , colectivos y avenidas principales . Expensas aproximadas de $45.000 por mes . Dep ...

ENTIDADES ANOTADAS:
  ORIENTACION    -> vista abierta al oeste
  ANTIGUEDAD     -> a estrenar
  EXPENSAS       -> $45.000
  ESTADO         -> a estrenar
  AMENITY        -> parrilla
  AMENITY        -> balcón


## Fine-tuning

Ambos modelos parten de **BETO** (`dccuchile/bert-base-spanish-wwm-cased`), un BERT entrenado
en español, y se ajustan a cada tarea.

La configuración está condicionada por el hardware disponible — una **GTX 1650 de 4 GB**:
`fp16` para reducir memoria, batch chico, y *gradient accumulation* para recuperar un batch
efectivo razonable sin ocupar más VRAM.

In [8]:
for tarea, key in [("NER", "ner"), ("Clasificacion", "classifier")]:
    c = cfg[key]
    print(f"--- {tarea} ---")
    print(f"  batch={c['batch_size']} x grad_accum={c['grad_accum']} "
          f"-> batch efectivo {c['batch_size'] * c['grad_accum']}")
    print(f"  epochs={c['epochs']}  lr={c['lr']}  max_length={c['max_length']}  fp16={c['fp16']}")

--- NER ---
  batch=4 x grad_accum=4 -> batch efectivo 16
  epochs=5  lr=3e-05  max_length=256  fp16=True
--- Clasificacion ---
  batch=4 x grad_accum=4 -> batch efectivo 16
  epochs=5  lr=2e-05  max_length=256  fp16=True


### Avisos largos: ventanas deslizantes

BETO tiene un límite **arquitectónico** de 512 sub-tokens — son las posiciones que aprendió durante
su pre-entrenamiento, no un parámetro que se pueda subir en un archivo de configuración.

Los avisos reales de CABA llegan a **1.173 sub-tokens**. Es decir que ni llevando el modelo a su
máximo se leerían enteros: con `max_length=512` entrarían completos 86 de 105.

La solución es partir cada aviso en ventanas que sí entren, pasarlas de a una y unir las
predicciones ([`src/models/chunking.py`](../src/models/chunking.py)). Dos detalles hacen que
funcione y no sea un corte burdo:

1. **Las ventanas se solapan.** Cortar en seco partiría al medio una entidad que caiga justo en el
   borde (`balcón | aterrazado`), y ninguna ventana la vería completa.
2. **Al unir, gana la ventana donde la palabra está más al centro**, porque ahí tiene contexto de
   los dos lados. Una palabra pegada al borde de una ventana suele caer en el medio de la siguiente.

Para clasificación se toma el **máximo** por clase entre ventanas: si la señal aparece en alguna
parte del aviso, el aviso la tiene. Promediar la diluiría — una mención de «dueño directo» quedaría
ahogada en un aviso de 4.000 caracteres.

> Por eso `max_length` de acá arriba es el tamaño de **ventana**, no un límite del aviso.

El entrenamiento se lanza fuera del notebook (tarda varios minutos en GPU):

```bash
python -m src.models.train_ner
python -m src.models.train_classifier
```

---
# Métricas: definición y justificación

### NER — F1 por entidad con `seqeval`, **no** accuracy por token

Dos razones:

1. **El accuracy por token miente.** La enorme mayoría de los tokens son `O` (texto sin entidad),
   así que un modelo que no detecte nada igual saca un accuracy altísimo.
2. **Lo que importa es el span completo.** Si la anotación dice `a estrenar` y el modelo marca sólo
   `estrenar`, para el uso posterior (armar la variable *estado de la propiedad*) eso está mal.
   `seqeval` cuenta una entidad como correcta sólo si coinciden **el tipo y los límites exactos**,
   que es el criterio estricto y el que corresponde acá.

### Clasificación — F1 **macro** además del micro

Las clases están desbalanceadas (ver la distribución de arriba). El F1 micro queda dominado por
las clases frecuentes; el **macro promedia las cuatro clases con igual peso**, así que penaliza
que el modelo ignore una clase rara. Y justamente las señales raras —`URGENCIA`, `OPORTUNIDAD`—
son las más interesantes para detectar subvaluación, que es el propósito del proyecto.

Se reporta también el F1 **por clase**, porque el promedio solo esconde en cuál falla.

In [9]:
# Metricas de test guardadas por el entrenamiento
for tarea, d in [("NER", cfg["ner"]["out_dir"]), ("CLS", cfg["classifier"]["out_dir"])]:
    p = ROOT / d / "test_metrics.json"
    if p.exists():
        m = json.loads(p.read_text(encoding="utf-8"))
        print(f"--- {tarea} (test) ---")
        for k, v in m.items():
            if isinstance(v, float) and not k.startswith("eval_runtime"):
                print(f"  {k:34s} {v:.4f}")
    else:
        print(f"--- {tarea}: sin entrenar todavia (falta {p}) ---")

--- NER (test) ---
  eval_loss                          0.0016
  eval_precision                     0.9941
  eval_recall                        0.9956
  eval_f1                            0.9948
  eval_samples_per_second            10.1980
  eval_steps_per_second              2.5500
  epoch                              5.0000
--- CLS (test) ---
  eval_loss                          0.0185
  eval_precision_macro               1.0000
  eval_recall_macro                  1.0000
  eval_f1_macro                      1.0000
  eval_f1_micro                      1.0000
  eval_f1_DUENO_DIRECTO              1.0000
  eval_f1_OPORTUNIDAD                1.0000
  eval_f1_URGENCIA                   1.0000
  eval_f1_REFACCION                  1.0000
  eval_samples_per_second            10.2430
  eval_steps_per_second              2.5610
  epoch                              5.0000


In [10]:
# Reportes detallados por entidad / por clase
for r in sorted(Path("reports").glob("*.md")):
    print("=" * 70)
    print(r.read_text(encoding="utf-8"))

# Evaluacion — Clasificacion multilabel

- **Modelo:** `models/cls-beto`
- **Conjunto evaluado:** `data/annotated/real_cls.jsonl` (real)
- **Ejemplos:** 105

## Reporte por clase

```
               precision    recall  f1-score   support

DUENO_DIRECTO     0.5200    0.2281    0.3171        57
  OPORTUNIDAD     0.6552    0.6129    0.6333        31
     URGENCIA     0.1111    0.0303    0.0476        33
    REFACCION     1.0000    0.0952    0.1739        21

    micro avg     0.5385    0.2465    0.3382       142
    macro avg     0.5716    0.2416    0.2930       142
 weighted avg     0.5255    0.2465    0.3023       142
  samples avg     0.2492    0.1302    0.1648       142
```

## Resumen

```json
{
  "f1_macro": 0.2929846487905873,
  "f1_micro": 0.33816425120772947,
  "n_ejemplos": 105
}
```

# Evaluacion — Clasificacion multilabel

- **Modelo:** `models/cls-beto`
- **Conjunto evaluado:** `data/annotated/cls_test.jsonl` (sintetico)
- **Ejemplos:** 120

## Reporte por clase

```
       

---
# Generalización sintético → real

**Este es el resultado más importante del trabajo, y conviene leerlo con cuidado.**

El F1 sobre el test sintético es altísimo. Eso *no* significa que el modelo sea excelente:
significa que **el dataset sintético es fácil**. Los avisos se generan a partir de plantillas,
así que el modelo puede aprender la plantilla en lugar del concepto.

La prueba honesta es correr el modelo sobre las descripciones **reales** scrapeadas de ZonaProp,
que son mucho más largas y tienen prosa desordenada, abreviaturas y direcciones.

In [11]:
from src.models.infer import extract_entities
from collections import Counter

reales = list(read_jsonl(ROOT / "data/raw/sample_real_caba.jsonl"))
sinteticos = list(read_jsonl(ROOT / "data/synthetic/listings.jsonl"))

largo_real = sum(len(r["description"]) for r in reales) // len(reales)
largo_sint = sum(len(r["description"]) for r in sinteticos) // len(sinteticos)
por_fuente = Counter(r["source"] for r in reales)

print(f"Avisos reales scrapeados  : {len(reales)}  {dict(por_fuente)}")
print(f"Largo medio de descripcion: real={largo_real} vs sintetico={largo_sint} caracteres")

# Las metricas comparadas, ya calculadas por src/models/evaluate.py
print("\n--- F1 sobre cada conjunto ---")
for tarea, etiqueta in [("ner", "NER (micro, seqeval)"), ("cls", "Clasificacion (macro)")]:
    clave = "f1_micro" if tarea == "ner" else "f1_macro"
    fila = []
    for conj in ["sintetico", "real"]:
        p = ROOT / f"reports/{tarea}_{conj}.json"
        v = json.loads(p.read_text(encoding="utf-8"))[clave] if p.exists() else None
        fila.append(f"{conj}={v:.3f}" if v is not None else f"{conj}=(sin datos)")
    print(f"  {etiqueta:24s} {'  |  '.join(fila)}")

Avisos reales scrapeados  : 105  {'argenprop': 80, 'zonaprop': 25}
Largo medio de descripcion: real=1574 vs sintetico=592 caracteres

--- F1 sobre cada conjunto ---


  NER (micro, seqeval)     sintetico=0.995  |  real=0.197
  Clasificacion (macro)    sintetico=1.000  |  real=0.293


In [12]:
# Inspeccion cualitativa: que marca el modelo sobre un aviso real concreto
if (ROOT / cfg["ner"]["out_dir"]).exists():
    r = reales[0]
    print("DESCRIPCION REAL:")
    print(" ", r["description"][:380], "...\n")
    print("ENTIDADES QUE DETECTA EL MODELO:")
    for e in extract_entities(r["description"][:900], cfg["ner"]["out_dir"], cfg["ner"]["max_length"]):
        print(f"    {e['type']:14s} -> {e['text']}")
else:
    print("Modelos no entrenados todavia.")

DESCRIPCION REAL:
  TORRE BELLINI! Piso alto de revista! Balcón con espectacular vista panorámica a la ciudad y al río! Todo a nuevo! Full amenities! 2 Coch! ÚNICO! “Torre bellini”
de revista!
Impecable piso muy alto!
Balcón con espectacular vista panorámica a la ciudad y al río!
Muy luminoso!
Todo luz y sol!
Excelente distribución!
Todo externo!
Doble circulación!
Todo a nuevo!
Palier privado!
Ha ...

ENTIDADES QUE DETECTA EL MODELO:


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

    ESTADO         -> a nuevo
    ESTADO         -> a nuevo
    AMENITY        -> piscina climatizada
    AMENITY        -> solárium
    AMENITY        -> jacuzzi
    AMENITY        -> solárium
    AMENITY        -> gimnasio
    AMENITY        -> vestuarios
    AMENITY        -> sauna
    AMENITY        -> sala de relax
    AMENITY        -> sum
    AMENITY        -> parrilla
    AMENITY        -> juegos para niños
    AMENITY        -> espacio verde
    AMENITY        -> microcine
    AMENITY        -> jaulas de golf
    AMENITY        -> laundry


### Qué aprendimos de esto

La primera versión del generador producía descripciones de ~220 caracteres, sin tildes y siempre
con la misma estructura. El modelo alcanzaba F1 = 1.0 sobre ese test, pero al aplicarlo a texto
real etiquetaba como `AMENITY` prácticamente cualquier sustantivo — «universidades», «avenidas»,
«ventilación» — y llegaba a marcar el nombre de una calle como `ORIENTACION`.

El diagnóstico: el modelo no había aprendido *qué es un amenity*, sino *dónde suele aparecer uno*
dentro de la plantilla. Nunca había visto un sustantivo que **no** fuera entidad.

Sobre esa base se rehízo el generador:

| Cambio | Por qué |
|---|---|
| Tildes en todo el vocabulario | Los avisos reales las usan y BETO es un modelo *cased*: distingue `balcón` de `balcon` |
| Oraciones distractoras sin entidades | Para que aprenda a predecir `O`; los tokens `O` pasaron a ser el 90% |
| Orden de secciones barajado | Si el orden es fijo, el modelo aprende la posición en vez del contenido |
| Frases de entrada variadas | No siempre «Cuenta con» antes de los amenities |
| Descripciones más largas | ~586 caracteres, más cerca de los ~1.600 reales |

**La conclusión no se maquilla:** los datos sintéticos sirven para validar que el pipeline
funciona de punta a punta, pero **no sustituyen anotación real**. Un F1 perfecto sobre datos
generados por uno mismo dice más sobre lo fácil que es el test que sobre la calidad del modelo.

---
# El producto final: texto libre → variables estructuradas

Esto es la salida de valor del proyecto. Una descripción cualquiera entra como texto y sale
convertida en atributos que un modelo de valuación podría consumir como *features*.

In [13]:
from src.models.infer import enrich

EJEMPLOS = [
    "Excelente 3 ambientes a estrenar al frente, con balcon aterrazado, pileta y cochera. "
    "Expensas $95.000. Dueno directo, escucho ofertas.",
    "Departamento a reciclar en Almagro, contrafrente, 40 anios de antiguedad. "
    "Necesita refaccion integral. Venta urgente por mudanza.",
]

if (ROOT / cfg["ner"]["out_dir"]).exists():
    for t in EJEMPLOS:
        out = enrich(t, cfg)
        print("TEXTO:", t[:90], "...")
        print("  entidades:")
        for e in out["entities"]:
            print(f"    {e['type']:14s} -> {e['text']}")
        print(f"  senales  : {out['signals'] or '(ninguna)'}\n")
else:
    print("Modelos no entrenados todavia. Corre:  scripts\\run_demo.bat --quick")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

TEXTO: Excelente 3 ambientes a estrenar al frente, con balcon aterrazado, pileta y cochera. Expen ...
  entidades:
    ESTADO         -> a
    ORIENTACION    -> al frente
    AMENITY        -> balcon aterrazado
    AMENITY        -> pileta
    AMENITY        -> cochera
    EXPENSAS       -> $95.000
  senales  : ['OPORTUNIDAD']



Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

TEXTO: Departamento a reciclar en Almagro, contrafrente, 40 anios de antiguedad. Necesita refacci ...
  entidades:
    ESTADO         -> a reciclar
    ORIENTACION    -> contrafrente
    ESTADO         -> refaccion integral
  senales  : ['URGENCIA', 'REFACCION']



---
# Limitaciones

Se documentan explícitamente, como pedía la propuesta:

**1. El anti-bot fue el límite real del scrape.** ZonaProp protege el sitio con Cloudflare, y la
evidencia mostró que el corte es **por sesión**, desde la segunda request: la misma URL que devolvía
30 avisos en una prueba aislada devolvía 0 dentro de una corrida. Subir los delays a 40 segundos no
cambió nada. Se descartó evadir la protección —proxies rotativos, *fingerprints* falsos, resolución
de CAPTCHA— porque eso ya no es consultar el sitio de otra manera sino saltear un control de
seguridad de un tercero.

La salida fue **cambiar de fuente, no de método**: Argenprop habilita el acceso automatizado en su
`robots.txt` (`Allow` hasta `?pagina-10`, `Disallow` de ahí en más), tope que se valida en el
código. Aun así el volumen quedó muy por debajo de los 8.000–15.000 avisos de la propuesta.

**2. Datos sintéticos para entrenar.** Los modelos se entrenaron sobre un generador con etiquetas
*gold por construcción*, y los avisos reales anotados se usan como evaluación externa. Como se vio
más arriba, el F1 perfecto sobre sintético **no** se traslada a texto real.

**3. Anotación semiautomática.** El pre-anotador es un LLM local (`llama3.2:3b`), elegido por la
restricción de 4 GB de VRAM. Sus errores se propagan a las etiquetas: por eso la revisión manual.
Mientras esa revisión no esté hecha, el F1 «real» mide **concordancia con otro modelo**, no con
verdad de referencia humana.

**4. Longitud de los avisos: resuelto.** BETO tiene un tope arquitectónico de 512 sub-tokens y los
avisos llegan a 1.173. Se resolvió con ventanas deslizantes (`src/models/chunking.py`), así que hoy
se lee el 100% del texto — pero es una capa extra de complejidad que un modelo de contexto largo
evitaría.

**5. Términos de uso.** Scraping con fines académicos, a ritmo respetuoso, sin redistribuir el
contenido del portal: el repositorio incluye una muestra acotada de datos ya estructurados, no
volcados de páginas.

# Trabajo futuro

Las variables generadas por esta capa de NLP están pensadas para integrarse, en el marco de la
**tesis de la maestría**, como *features* de un modelo hedónico de valuación, para detectar activos
subvaluados en CABA. Esa integración excede el alcance de este Trabajo Final.

El paso inmediato, y el que más movería la aguja, es **anotar a mano un conjunto real grande**: es
lo que separa este pipeline funcionando de un modelo que se pueda usar de verdad.